# zi2zi-JiT · 字体训练 CloudStudio Notebook（一键版）

基于 **zi2zi-JiT**（ICW fork）。参照 HanziGen 的 `hanzigen_cloudstudio.ipynb` 设计：

- 训练所需参数全部集中在 **Cell 0**，其余代码 Cell 不用改
- 每个代码 Cell 前都有 **markdown 说明**（作用 / 输入 / 输出 / 产物位置）
- 导入（预训练模型、目标字体）与导出（PNG zip 下载）遵循腾讯云**可视化目录**约定

**使用流程：**

1. 把预训练模型 `zi2zi-JiT-B-16.pth`（README 下载链接）上传到 `models/`
2. 把要学的新字体（.ttf/.otf）上传到 `fonts/`
3. 修改 **Cell 0**（其余 Cell 从头到尾依次运行即可）
4. 跑完后在 `exports/` 或 Cell 6 的下载按钮拿到 zip

**目录约定（腾讯云"可视化 / 不可视化目录"）：**

| 目录 | 类型 | 内容 |
|---|---|---|
| `fonts/` | ✅ 可视化 | 目标字体（上传）、参照字体（上传） |
| `models/` | ✅ 可视化 | 预训练模型（上传） |
| `data/` | ✅ 可视化 | 自动生成的数据集 |
| `outputs/` | ✅ 可视化 | 训练产物 + 推理 PNG + 补集 PNG |
| `exports/` | ✅ 可视化 | 最终打包 zip（可直接下载） |
| `/root`、`/tmp`、conda 缓存等 | ❌ 不可视化 | 文件树看不到，实例回收会丢，**不要放重要数据** |

> 建议先在 Cloud Studio 里按 `environment.yaml` 建好 Python 3.10 + PyTorch 环境；
> Cell 1 也会自动检查并补装依赖（首次运行较慢）。

## Cell 0 · 参数预设（训练新字体只改这一个 Cell）

> 每个参数都标了 **✅建议改 / ⛔不建议改 / 🤖自动计算**。
> 带 `*` 的"须与预训练模型一致"，改了会导致加载预训练模型失败。

| 参数 | 作用 | 建议 |
|---|---|---|
| **运行开关** | | |
| `DO_DATA_PREP` | 是否重新生成数据集 | ✅ 首次/换字体=True；已有数据集可设 False 跳过 |
| `DO_TRAIN` | 是否训练 | ✅ 同上 |
| `DO_GENERATE` | 是否推理生成 PNG | ✅ 同上 |
| `DO_MISSING_GEN` | 是否生成"缺失字补集" | ✅ 同上（补集=按 CHARSET 补齐目标字体缺的字） |
| `DO_EXPORT` | 是否打包导出 | ✅ 同上 |
| **导入（素材放可视化目录）** | | |
| `TARGET_FONTS` | **要学的新字体** | ✅ **必改**（可多个=多字体风格） |
| `SOURCE_FONT` | 参照字体，提供 content 字形 | ✅ 留空自动挑一个与目标不同的字体 |
| `BASE_CHECKPOINT` | 预训练模型路径 | ✅ **两个模型（B/L）二选一**：下载哪个就写哪个文件名，并同步改 `MODEL` 与 `CFG`（B→`JiT-B/16`+2.6，L→`JiT-L/16`+2.4），脚本不会自动切换 |
| `FONTS_DIR` … `EXPORTS_DIR` | 各目录 | ⛔ 不建议改 |
| **数据准备** | | |
| `CHARSET` | 字符集：简中`gb2312`（6763 字）/ 繁中`big5` / 日文`jisx0208` / 韩文`ksx1001` / **真 GBK：`gbk`（20,902 字，同 HanziGen，已内置）** | ✅ 按你的字体面向地区改。用真 GBK 补字请同步 `NUM_CHARS=30000`、`MAX_CHARS_PER_FONT=None`（训练会更久） |
| `TRAIN_CHARS_PER_FONT` | 每个字体从 CHARSET 抽多少个字符进**训练集** | ✅ 字符越多风格学得越全；补集效果最佳 → **≥ 字符集大小**（gb2312 用 6763） |
| `TEST_CHARS_PER_FONT` | 每个字体抽多少个**训练未见过**的字符进**测试集**（只评估泛化，不参与训练） | ⛔ 8 够用 |
| `RESOLUTION` | 渲染分辨率 | ⛔ 须与 `IMG_SIZE` 一致（256） |
| `TRAIN_SEED` / `TEST_SEED` | 随机种子 | ⛔ 复现用 |
| `NUM_WORKERS_DATA_PREP` | 数据生成并行数 | ⛔ 不建议改 |
| **LoRA 训练** | | |
| `MODEL`* | JiT-B/16 或 JiT-L/16 | ⛔ 须与预训练模型一致 |
| `IMG_SIZE`* | 训练分辨率 | ⛔ 须与预训练模型一致（256） |
| `NUM_FONTS`* | 字体嵌入维度 | ⛔ 必须 = 预训练模型（1000），改了就加载失败 |
| `NUM_CHARS` | 字符嵌入上界（只需 ≥ 数据集字符数） | ✅ 默认 20000 够用；`CHARSET="gbk"` 时**必须** 30000 |
| `MAX_CHARS_PER_FONT` | 训练时每字体实际使用字符数的上限（None=全部） | ✅ 补集建议 `None`（让 TRAIN_CHARS_PER_FONT 抽的字全部参与训练） |
| `LORA_R` / `LORA_ALPHA` | LoRA 容量（决定新字体学得多细） | ✅ **风格化明显→64；规整→默认32**；显存几乎不受影响 |
| `LORA_TARGETS` / `LORA_DROPOUT` / `PROJ_DROPOUT` | LoRA 注入层 / 丢弃率 | ⛔ 不建议改 |
| `EPOCHS` | 训练轮数 | ✅ **风格化明显→300+；规整→200 足够** |
| `BLR` / `MIN_LR` / `WARMUP_EPOCHS` | 学习率 | ⛔ README 推荐值 |
| `SAVE_LAST_FREQ` | 保存 checkpoint 频率 | ⛔ 不建议改 |
| `P_MEAN` / `P_STD` / `NOISE_SCALE` | 扩散噪声分布（**训练期参数，固化进 checkpoint，生成阶段不能覆盖**） | ⛔ 一般不调。EDM 经验：笔画乱/错字多→`NOISE_SCALE` 降到 0.8；模糊/缺细节→提到 1.2；`P_MEAN`/`P_STD` 保持 -0.8/0.8 |
| `CFG` | 引导强度（越大风格越浓，过大会糊） | ✅ **风格化明显→3.5~4.0；默认 2.6**（JiT-L/16 用 2.4） |
| `ONLINE_EVAL` / `EVAL_STEP_FOLDERS` / `EVAL_FREQ` / `NUM_IMAGES` | 训练中在线评估 | ⛔ 不建议改 |
| `BATCH_SIZE` / `GEN_BSZ` | 训练/推理批量 | 🤖 **自动计算**（AUTO_TUNE=True 时被覆盖，不用手动改） |
| **硬件自动调优** | | |
| `AUTO_TUNE` | 自动推算 batch_size 等 | ✅ 默认 True 即可 |
| `TUNE_METHOD` | `probe`=运行时实测（最准） / `table`=标定表估算 | ✅ 推荐 `probe` |
| `TUNE_RESERVE` / `TUNE_MAX_*` | 调优安全系数/上限 | ⛔ 不建议改 |
| **推理生成** | | |
| `GENERATE_*` | 生成张数/批量/CFG/采样方法 | ✅ **效果不好时调**：`GENERATE_CFG` 加大、`GENERATE_SAMPLING_METHOD="heun"`、加大步数 |
| **缺失字补集** | | |
| `DO_MISSING_GEN` / `MISSING_*` | 补集开关与参数 | ✅ 需要补缺字时开；`MISSING_CHARSET` 默认沿用 `CHARSET` |
| **导出** | | |
| `EXPORT_PREFIX` / `EXPORT_INCLUDE_CHECKPOINT` | zip 名前缀 / 是否含 checkpoint | ⛔ 不建议改 |

**常见场景怎么调：**

| 场景 | 建议 |
|---|---|
| **风格化明显的字体**（行书/草书/手写体/装饰体） | `LORA_R=64`、`EPOCHS=300+`、`CFG=3.5~4.0`、`TRAIN_CHARS_PER_FONT` 尽量覆盖整个 CHARSET |
| **字形规整的字体**（黑体/宋体/楷体） | 默认参数即可，`CFG` 保持 2.6 |
| **生成效果不理想** | 优先试：加大 `GENERATE_CFG` → 换 `heun` 采样 → 加步数 → 加 `EPOCHS`/`LORA_R` |
| **显存不够 / 想跑得快** | 保持 `AUTO_TUNE=True`（会自动收紧 batch）；不要手动把 batch 调大 |
| **只想要补集（缺字补全）** | `DO_TRAIN/DO_GENERATE` 可留 True（需要先训练），补集产物在 `outputs/<字体>/missing_chars/` |

**关于《腾讯云可用配置及建议260819.txt》：** 自动调优负责"**在这台机器上 batch_size 用多少**"；该文档负责"**选哪台机器**"（T4 / V100 / A10 的显存、价格、三阶段选型：阶段一数据准备低配、阶段二训练、阶段三导出）。所以**具体 batch 数值不用再翻文档**（已自动计算），但**选机型时仍可参考文档**。

In [ ]:
# ============================================================
# Cell 0 · 参数预设（训练新字体只需修改本 Cell）
# 每个参数的作用 / 是否建议修改，见上方 markdown 说明表
# ============================================================
import os

# ---------- 运行开关 ----------
DO_DATA_PREP   = True
DO_TRAIN       = True
DO_GENERATE    = True
DO_MISSING_GEN = True     # 缺失字补集（按 CHARSET 补齐目标字体缺失的字）
DO_EXPORT      = True

# ---------- 导入（素材放可视化目录） ----------
FONTS_DIR       = os.path.join(os.path.abspath(""), "fonts")
MODELS_DIR      = os.path.join(os.path.abspath(""), "models")
DATA_DIR        = os.path.join(os.path.abspath(""), "data")
OUTPUTS_DIR     = os.path.join(os.path.abspath(""), "outputs")
EXPORTS_DIR     = os.path.join(os.path.abspath(""), "exports")

SOURCE_FONT     = ""                                # 参照字体（留空自动挑）
TARGET_FONTS    = ["fonts/MyTargetFont.ttf"]        # <-- 必改
BASE_CHECKPOINT = "models/zi2zi-JiT-B-16.pth"

# ---------- 数据准备 ----------
CHARSET               = "gb2312"   # gb2312 / gbk / big5 / jisx0208 / ksx1001 / simple_chars
TRAIN_CHARS_PER_FONT  = 500        # 建议 >= 字符集大小（gb2312 用 6763），补集效果最佳
TEST_CHARS_PER_FONT   = 8
RESOLUTION            = 256
TRAIN_SEED            = 42
TEST_SEED             = 99999
NUM_WORKERS_DATA_PREP = 4

# ---------- LoRA 训练 ----------
MODEL              = "JiT-B/16"   # 与预训练模型一致，勿改
IMG_SIZE           = 256
NUM_FONTS          = 1000         # 与预训练模型一致，勿改
NUM_CHARS          = 20000        # 与预训练模型一致，勿改
MAX_CHARS_PER_FONT = 200          # None=全部（补集建议 None）
LORA_R             = 32           # 风格化明显 -> 64
LORA_ALPHA         = 32
LORA_TARGETS       = "qkv,proj,w12,w3"
LORA_DROPOUT       = 0.0
PROJ_DROPOUT       = 0.1
EPOCHS             = 200          # 风格化明显 -> 300+
BLR                = 8e-4
MIN_LR             = 1e-6
WARMUP_EPOCHS      = 1
SAVE_LAST_FREQ     = 10
SEED               = 42
P_MEAN             = -0.8
P_STD              = 0.8
NOISE_SCALE        = 1.0
CFG                = 2.6          # 风格化明显 -> 3.5~4.0
ONLINE_EVAL        = True
EVAL_STEP_FOLDERS  = True
EVAL_FREQ          = 10
NUM_IMAGES         = 6
BATCH_SIZE         = 16           # AUTO_TUNE=True 时会被自动覆盖
GEN_BSZ            = 16

# ---------- 硬件自动调优 ----------
AUTO_TUNE           = True
TUNE_METHOD         = "probe"     # probe=实测 / table=标定表
TUNE_RESERVE        = 0.85
TUNE_MAX_BATCH      = 128
TUNE_MAX_GEN_BSZ    = 32
TUNE_NUM_WORKERS_CAP = 12

# ---------- 推理生成 ----------
GENERATE_NUM_IMAGES         = None
GENERATE_BATCH_SIZE         = 64
GENERATE_CFG                = None
GENERATE_SAMPLING_METHOD    = None
GENERATE_NUM_SAMPLING_STEPS = None
GENERATE_PAIRWISE           = None

# ---------- 缺失字补集 ----------
MISSING_CHARSET         = None           # None=沿用 CHARSET
MISSING_BATCH_SIZE      = 32
MISSING_CFG             = None
MISSING_SAMPLING_METHOD = None
MISSING_NUM_SAMPLING_STEPS = None
MISSING_NUM_IMAGES      = None           # None=全部缺失字
MISSING_PAIRWISE        = "src_gen"      # 输出 源字形|生成结果 对比图
MISSING_REF_CHARS       = ""             # 逗号分隔样式参考字（留空自动挑）

# ---------- 导出 ----------
EXPORT_PREFIX             = "zi2zi_jit"
EXPORT_INCLUDE_CHECKPOINT = True

# ---------- 派生路径（自动计算，不用改） ----------
FONT_TAG       = os.path.splitext(os.path.basename(TARGET_FONTS[0]))[0]
DATASET_DIR    = os.path.join(DATA_DIR, FONT_TAG)
TRAIN_DIR      = os.path.join(DATASET_DIR, "train")
TEST_NPZ_PATH  = os.path.join(DATASET_DIR, "test.npz")
OUTPUT_DIR     = os.path.join(OUTPUTS_DIR, FONT_TAG)
GEN_OUTPUT_DIR = os.path.join(OUTPUT_DIR, "generated_chars")
MISSING_OUTPUT_DIR = os.path.join(OUTPUT_DIR, "missing_chars")

## Cell 1 · 环境初始化 + 导入检查 + 硬件自动调优

**本 Cell 做什么（无需修改）：**

1. 直接把项目根目录设为 ipynb 所在目录（**不要在腾讯云里 git clone**，clone 出来的目录会不可视化）
2. 检查依赖（缺 torch 时自动 `pip install`，首次运行较慢）
3. 导入检查：目标字体是否存在、自动挑选源字体、预训练模型是否存在（缺什么会报错提示）
4. **配置自检**：`MODEL`↔预训练模型尺寸是否配套、`CHARSET`/`NUM_CHARS`/`MAX_CHARS_PER_FONT`
   等参数语义是否合理、已有 `data/` 缓存与当前 `CHARSET` 是否一致（发现错误会中断本 Cell 并列出所有问题）
5. 硬件自动调优：按 `AUTO_TUNE`/`TUNE_METHOD` 自动推算 `BATCH_SIZE`/`GEN_BSZ`/`NUM_WORKERS`，
   并打印出来供 Cell 3 使用

**产物：** 输出调优后的 `BATCH_SIZE`、`GEN_BSZ`、`NUM_WORKERS` 三个变量。

> 建议：把整个项目（含本 ipynb）上传/解压后在 Cloud Studio 里**直接打开**，所有目录即可视化，不需要再 clone。

In [ ]:
# ============================================================
# Cell 1 · 环境初始化 + 导入检查 + 硬件自动调优（一般不用改）
# ============================================================
import os, sys, glob, shutil, subprocess, datetime, zipfile, io, base64

# 1) 项目根目录 = 本 ipynb 所在目录（不要再 git clone，腾讯云 clone 的目录会不可视化）
PROJECT_ROOT = os.path.abspath("")  # 在项目里直接打开本 ipynb，工作目录即项目根目录
os.chdir(PROJECT_ROOT)
sys.path.insert(0, PROJECT_ROOT)
print("[Cell1] 项目根目录:", PROJECT_ROOT)

# 2) 依赖检查（缺 torch 才安装）
try:
    import torch, torchvision, numpy  # noqa
except ImportError:
    print("[Cell1] 安装依赖（首次约需几分钟）...")
    subprocess.run([sys.executable, "-m", "pip", "install",
                    "torch==2.5.1", "torchvision==0.20.1", "numpy", "opencv-python",
                    "timm", "tensorboard", "scipy", "einops", "gdown", "fonttools",
                    "Pillow", "pytorch-msssim", "lpips", "tqdm", "matplotlib"], check=True)

import torch
print("[Cell1] torch:", torch.__version__, "| CUDA:", torch.cuda.is_available())

# 3) 按 PROJECT_ROOT 重新定位路径（覆盖 Cell 0 中的相对值）
FONTS_DIR   = os.path.join(PROJECT_ROOT, "fonts")
MODELS_DIR  = os.path.join(PROJECT_ROOT, "models")
DATA_DIR    = os.path.join(PROJECT_ROOT, "data")
OUTPUTS_DIR = os.path.join(PROJECT_ROOT, "outputs")
EXPORTS_DIR = os.path.join(PROJECT_ROOT, "exports")
FONT_TAG       = os.path.splitext(os.path.basename(TARGET_FONTS[0]))[0]
DATASET_DIR    = os.path.join(DATA_DIR, FONT_TAG)
TRAIN_DIR      = os.path.join(DATASET_DIR, "train")
TEST_NPZ_PATH  = os.path.join(DATASET_DIR, "test.npz")
OUTPUT_DIR     = os.path.join(OUTPUTS_DIR, FONT_TAG)
GEN_OUTPUT_DIR = os.path.join(OUTPUT_DIR, "generated_chars")
MISSING_OUTPUT_DIR = os.path.join(OUTPUT_DIR, "missing_chars")
for d in (FONTS_DIR, MODELS_DIR, DATA_DIR, OUTPUTS_DIR, EXPORTS_DIR):
    os.makedirs(d, exist_ok=True)

# 4) 导入检查：目标字体 / 源字体 / 预训练模型
for t in TARGET_FONTS:
    p = t if os.path.isabs(t) else os.path.join(PROJECT_ROOT, t)
    assert os.path.exists(p), "[Cell1] 目标字体不存在: %s（请上传到 %s）" % (p, FONTS_DIR)
print("[Cell1] 目标字体 OK:", TARGET_FONTS)

def resolve_source_font():
    if SOURCE_FONT:
        p = SOURCE_FONT if os.path.isabs(SOURCE_FONT) else os.path.join(PROJECT_ROOT, SOURCE_FONT)
        assert os.path.exists(p), "[Cell1] 源字体不存在: " + p
        return p
    target_names = {os.path.basename(t) for t in TARGET_FONTS}
    fonts = []
    for ext in ("*.ttf", "*.otf", "*.ttc"):
        fonts += glob.glob(os.path.join(FONTS_DIR, ext))
    cand = [f for f in fonts if os.path.basename(f) not in target_names]
    assert cand, "[Cell1] 未找到源字体：请在 fonts/ 放一个参照字体，或设置 SOURCE_FONT"
    return sorted(cand)[0]

SOURCE_FONT_PATH = resolve_source_font()
print("[Cell1] 源字体:", SOURCE_FONT_PATH)

ckpt = BASE_CHECKPOINT if os.path.isabs(BASE_CHECKPOINT) else os.path.join(PROJECT_ROOT, BASE_CHECKPOINT)
assert os.path.exists(ckpt), "[Cell1] 预训练模型不存在: %s（请放到 %s）" % (ckpt, MODELS_DIR)
print("[Cell1] 预训练模型:", ckpt)

# 4.5) 配置自检：MODEL<->checkpoint 配套 / 字符集参数 / 数据缓存一致性
import json as _json
import re as _re
from data_processing.charsets import get_charset_codepoints

print("\n[Cell1] ---- 配置自检 ----")
_config_errors = []

def _cok(cond, msg):
    if cond:
        print("  [OK] " + msg)
    else:
        _config_errors.append(msg)
        print("  [ERR] " + msg)

def _cwarn(msg):
    print("  [!!] " + msg)

# a) MODEL <-> BASE_CHECKPOINT 尺寸配套（B-16.pth 配 JiT-B/16，L-16.pth 配 JiT-L/16）
_m_ck = _re.search(r"[_-]([BL])[_-]\d+", os.path.basename(ckpt))
_m_md = _re.search(r"([BL])/", MODEL) if isinstance(MODEL, str) else None
if _m_ck and _m_md:
    _cok(_m_ck.group(1) == _m_md.group(1),
         "checkpoint %s 尺寸=%s 与 MODEL=%s 配套" % (os.path.basename(ckpt), _m_ck.group(1), MODEL))
elif _m_ck and not _m_md:
    _cok(False, "MODEL='%s' 无法解析尺寸（应为 JiT-B/16 或 JiT-L/16）" % MODEL)
else:
    _cwarn("checkpoint 文件名不含 B/L 尺寸标记，跳过配套检查（load_state_dict 严格加载仍会兜底）")

# b) 字符集参数语义
try:
    _cps = get_charset_codepoints(CHARSET)
    _cs = len(_cps)
    print("  [OK] CHARSET=%s 共 %d 字" % (CHARSET, _cs))
except Exception as _e:
    _cps, _cs = None, 0
    _cok(False, "CHARSET='%s' 不受支持: %s" % (CHARSET, _e))

if _cps is not None:
    _cok(NUM_CHARS >= _cs, "NUM_CHARS(%d) >= 字符集大小(%d)" % (NUM_CHARS, _cs))
    if TRAIN_CHARS_PER_FONT > _cs:
        _cwarn("TRAIN_CHARS_PER_FONT(%d) > 字符集大小(%d)，实际会静默取全部 %d 字" % (TRAIN_CHARS_PER_FONT, _cs, _cs))
    if CHARSET == "gbk" and MAX_CHARS_PER_FONT is not None:
        _cwarn("真 GBK 补集建议 MAX_CHARS_PER_FONT=None（当前=%d），否则每字体仅 %d 字参与训练" % (MAX_CHARS_PER_FONT, MAX_CHARS_PER_FONT))
    if TRAIN_CHARS_PER_FONT >= _cs and MAX_CHARS_PER_FONT is not None and MAX_CHARS_PER_FONT < _cs:
        _cwarn("TRAIN_CHARS_PER_FONT 已覆盖全集，但 MAX_CHARS_PER_FONT=%d 会截断到 %d 字" % (MAX_CHARS_PER_FONT, MAX_CHARS_PER_FONT))

# c) 训练配置一致性
_cok(NUM_FONTS >= len(TARGET_FONTS),
     "NUM_FONTS(%d) >= 目标字体数(%d)" % (NUM_FONTS, len(TARGET_FONTS)))
_cok(RESOLUTION == IMG_SIZE, "RESOLUTION(%d) == IMG_SIZE(%d)" % (RESOLUTION, IMG_SIZE))

# d) 已有数据集缓存与 CHARSET 一致性（改了 CHARSET 后旧数据集不会被自动重建）
if os.path.isdir(TRAIN_DIR) or os.path.exists(TEST_NPZ_PATH):
    _old_cs = set()
    for _mf in glob.glob(os.path.join(TRAIN_DIR, "*", "metadata.json")):
        try:
            with open(_mf, encoding="utf-8") as _f:
                _old_cs.add(str(_json.load(_f).get("charset_filter", "")).lower())
        except Exception:
            pass
    if _old_cs:
        _same = _old_cs == {str(CHARSET).lower()}
        if DO_DATA_PREP:
            _cok(_same, "已有数据集 charset=%s 与当前 CHARSET=%s 一致" % (sorted(_old_cs), CHARSET))
        elif not _same:
            _cwarn("已有数据集 charset=%s 与当前 CHARSET=%s 不一致；DO_DATA_PREP=False 将用旧数据集" % (sorted(_old_cs), CHARSET))
        else:
            print("  [OK] 已有数据集 charset=%s 与当前 CHARSET=%s 一致" % (sorted(_old_cs), CHARSET))
    else:
        _cwarn("已有数据集但读不到 metadata.json（可能是旧版本），建议删除 %s 后重跑" % DATASET_DIR)
else:
    print("  [OK] 无已有数据集（首次运行）")

# e) 汇总：有错误则中断本 Cell
if _config_errors:
    print("\n[Cell1] !!! 配置自检发现 %d 个错误，请修正 Cell 0 后重跑本 Cell !!!" % len(_config_errors))
    for _i, _e in enumerate(_config_errors, 1):
        print("  %d) %s" % (_i, _e))
    raise RuntimeError("配置自检未通过：%d 个错误" % len(_config_errors))
print("[Cell1] 配置自检全部通过")
print()

# 5) 硬件自动调优：自动推算 BATCH_SIZE / GEN_BSZ / NUM_WORKERS
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

try:
    from util.auto_tune import auto_tune, probe_batch_size  # noqa
    HAS_AUTO_TUNE = True
except ImportError:
    HAS_AUTO_TUNE = False
    def auto_tune(**kw):
        total = torch.cuda.get_device_properties(0).total_memory / 1024**3 if torch.cuda.is_available() else 0
        per = 0.25 if kw.get("model_name", "JiT-B/16") == "JiT-B/16" else 0.5
        bsz = int(max(total * kw.get("reserve", 0.85) - 1.5, 0) / per) if total else kw.get("batch_size_fallback", 16)
        bsz = max(1, min(bsz, kw.get("max_batch", 128)))
        return {"batch_size": bsz,
                "gen_bsz": max(1, min(bsz, kw.get("max_gen_bsz", 32))),
                "num_workers": min(os.cpu_count() or 1, kw.get("num_workers_cap", 12)),
                "method": "fallback"}

if AUTO_TUNE:
    TUNE = auto_tune(method=TUNE_METHOD, model_name=MODEL, img_size=IMG_SIZE,
                     reserve=TUNE_RESERVE, max_batch=TUNE_MAX_BATCH,
                     max_gen_bsz=TUNE_MAX_GEN_BSZ, num_workers_cap=TUNE_NUM_WORKERS_CAP,
                     batch_size_fallback=BATCH_SIZE)
    BATCH_SIZE = TUNE["batch_size"]
    GEN_BSZ    = TUNE["gen_bsz"]
    NUM_WORKERS = TUNE["num_workers"]
    print("[Cell1] 自动调优完成（%s）-> batch_size:%d gen_bsz:%d num_workers:%d"
          % (TUNE.get("method", "?"), BATCH_SIZE, GEN_BSZ, NUM_WORKERS))
else:
    NUM_WORKERS = min(os.cpu_count() or 1, TUNE_NUM_WORKERS_CAP)
    print("[Cell1] AUTO_TUNE=False，使用固定 batch_size:", BATCH_SIZE)

print("[Cell1] 就绪，继续执行 Cell 2（数据准备）")

## Cell 2 · 数据准备（字体 -> `train/` `test/` `test.npz`）

**本 Cell 做什么（无需修改）：**

- 只把 `TARGET_FONTS` 指定的字体放进临时目录，调用 `scripts/generate_font_dataset.py`
- 渲染字形、按 `CHARSET` 过滤、抽训练/测试字、生成 `data/<字体>/` 下的数据集
- 已在已有数据集时自动跳过

**产物：** `data/<字体>/train/`（训练图）、`data/<字体>/test/`（测试图）、`data/<字体>/test.npz`（推理用）。

In [ ]:
# ============================================================
# Cell 2 · 数据准备（一般不用改）
# ============================================================
if DO_DATA_PREP:
    if os.path.exists(TRAIN_DIR) and os.path.exists(TEST_NPZ_PATH):
        print("[Cell2] 数据集已存在，跳过生成:", DATASET_DIR)
    else:
        staging = os.path.join(DATA_DIR, "_staging_fonts")
        os.makedirs(staging, exist_ok=True)
        for t in TARGET_FONTS:
            p = t if os.path.isabs(t) else os.path.join(PROJECT_ROOT, t)
            shutil.copy2(p, staging)
        cmd = [sys.executable, os.path.join(PROJECT_ROOT, "scripts", "generate_font_dataset.py"),
               "--source-font", SOURCE_FONT_PATH,
               "--font-dir", staging,
               "--output-dir", DATASET_DIR,
               "--train-chars-per-font", str(TRAIN_CHARS_PER_FONT),
               "--test-chars-per-font", str(TEST_CHARS_PER_FONT),
               "--resolution", str(RESOLUTION),
               "--charset", CHARSET,
               "--train-seed", str(TRAIN_SEED),
               "--test-seed", str(TEST_SEED),
               "--num-workers", str(NUM_WORKERS_DATA_PREP)]
        print("[Cell2] 命令:", " ".join(cmd))
        subprocess.run(cmd, check=True)
else:
    print("[Cell2] DO_DATA_PREP=False，跳过")

## Cell 3 · LoRA 训练

**本 Cell 做什么（无需修改）：**

- 用 Cell 0 的超参调用 `lora_single_gpu_finetune_jit.py`
- `AUTO_TUNE=True` 且 `TUNE_METHOD=probe` 时：会先构建一个与真实训练**完全一致**的模型
  （含 LoRA 注入），实测单样本显存，再精调 `batch_size` ——
  **改 `LORA_R`/模型变体/分辨率后都不用再手动猜显存**
- 训练中按 `EVAL_FREQ` 在线生成样例图（`outputs/<字体>/`）

**产物：** `outputs/<字体>/checkpoint-last.pth`（LoRA checkpoint，后续 Cell 都用它）。

> 训练轮数、CFG 等调整建议见 Cell 0 上方表格。

In [ ]:
# ============================================================
# Cell 3 · LoRA 训练（一般不用改）
# ============================================================
if DO_TRAIN:
    from lora_single_gpu_finetune_jit import get_args_parser, main as lora_main

    argv = [
        "--data_path", TRAIN_DIR,
        "--test_npz_path", TEST_NPZ_PATH,
        "--output_dir", OUTPUT_DIR,
        "--base_checkpoint", BASE_CHECKPOINT,
        "--model", MODEL,
        "--img_size", str(IMG_SIZE),
        "--num_fonts", str(NUM_FONTS),
        "--num_chars", str(NUM_CHARS),
        "--lora_r", str(LORA_R),
        "--lora_alpha", str(LORA_ALPHA),
        "--lora_targets", LORA_TARGETS,
        "--lora_dropout", str(LORA_DROPOUT),
        "--proj_dropout", str(PROJ_DROPOUT),
        "--epochs", str(EPOCHS),
        "--batch_size", str(BATCH_SIZE),
        "--blr", str(BLR),
        "--min_lr", str(MIN_LR),
        "--warmup_epochs", str(WARMUP_EPOCHS),
        "--save_last_freq", str(SAVE_LAST_FREQ),
        "--P_mean", str(P_MEAN),
        "--P_std", str(P_STD),
        "--noise_scale", str(NOISE_SCALE),
        "--cfg", str(CFG),
        "--num_images", str(NUM_IMAGES),
        "--seed", str(SEED),
        "--device", "cuda" if torch.cuda.is_available() else "cpu",
    ]
    if MAX_CHARS_PER_FONT is not None:
        argv += ["--max_chars_per_font", str(MAX_CHARS_PER_FONT)]

    args = get_args_parser().parse_args(argv)
    if ONLINE_EVAL:
        args.online_eval = True
    if EVAL_STEP_FOLDERS:
        args.eval_step_folders = True
    args.num_workers = NUM_WORKERS

    # probe 实测精调 batch_size（自动反映 LoRA / encoder / 模型变体对显存的影响）
    if AUTO_TUNE and TUNE_METHOD == "probe" and torch.cuda.is_available() and HAS_AUTO_TUNE:
        print("[Cell3] 构建探测模型（与真实训练一致，含 LoRA 注入）实测单样本显存 ...")
        import torch._dynamo
        torch._dynamo.config.cache_size_limit = 128
        from denoiser import Denoiser
        from util.lora_utils import (inject_lora, mark_only_lora_as_trainable,
                                     _is_lora_state_dict, resolve_checkpoint_path)

        probe_model = Denoiser(args)
        probe_model.update_ema = lambda: None
        ckpt_path = resolve_checkpoint_path(args.base_checkpoint)
        ck = torch.load(ckpt_path, map_location="cpu", weights_only=False)
        sd = ck["model"] if isinstance(ck, dict) and "model" in ck else ck
        is_lora = _is_lora_state_dict(sd)
        del ck
        if not is_lora:
            probe_model.load_state_dict(sd, strict=True)
        targets = [t.strip() for t in args.lora_targets.split(",") if t.strip()]
        inject_lora(probe_model.net, targets, r=args.lora_r, alpha=args.lora_alpha, dropout=args.lora_dropout)
        if is_lora:
            probe_model.load_state_dict(sd, strict=True)
        mark_only_lora_as_trainable(probe_model, train_font_emb=True)
        probe_model.to(device)

        safe, per_gb = probe_batch_size(probe_model, device, img_size=IMG_SIZE,
                                        num_fonts=NUM_FONTS, num_chars=NUM_CHARS,
                                        reserve=TUNE_RESERVE, max_batch=TUNE_MAX_BATCH)
        if safe is not None:
            print("[Cell3] 实测单样本显存: %.3f GB -> batch_size=%d" % (per_gb, safe))
            args.batch_size = safe
            args.gen_bsz = max(1, min(safe, TUNE_MAX_GEN_BSZ))
        del probe_model
        torch.cuda.empty_cache()

    print("[Cell3] 最终 batch_size:", args.batch_size, "| gen_bsz:", args.gen_bsz, "| num_workers:", args.num_workers)
    lora_main(args)
else:
    print("[Cell3] DO_TRAIN=False，跳过")

## Cell 4 · 推理生成 PNG（测试集字）

**本 Cell 做什么（无需修改）：**

- 用 `outputs/<字体>/checkpoint-last.pth` + `data/<字体>/test.npz` 调用 `generate_chars.py`
- 生成 test 集每个字的对比图（`src | generated`）与单字图
- `AUTO_TUNE=True` 时会按显存自动收紧推理批量

**产物：** `outputs/<字体>/generated_chars/`（`compare/` 对比图 + `generated/` 单字图）。

> 只针对"目标字体本来就有、且测试集抽中的字"。想补"目标字体缺失的字"请看 **Cell 5**。

In [ ]:
# ============================================================
# Cell 4 · 推理生成（一般不用改）
# ============================================================
if DO_GENERATE:
    ckpt = os.path.join(OUTPUT_DIR, "checkpoint-last.pth")
    assert os.path.exists(ckpt), "[Cell4] 未找到 checkpoint: %s" % ckpt

    from generate_chars import get_args_parser, main as gen_main
    args = get_args_parser().parse_args([
        "--checkpoint", ckpt,
        "--test_npz", TEST_NPZ_PATH,
        "--output_dir", GEN_OUTPUT_DIR,
        "--device", "auto",
    ])
    if GENERATE_NUM_IMAGES is not None:
        args.num_images = GENERATE_NUM_IMAGES
    if GENERATE_BATCH_SIZE is not None:
        args.batch_size = GENERATE_BATCH_SIZE
    if GENERATE_CFG is not None:
        args.cfg = GENERATE_CFG
    if GENERATE_SAMPLING_METHOD is not None:
        args.sampling_method = GENERATE_SAMPLING_METHOD
    if GENERATE_NUM_SAMPLING_STEPS is not None:
        args.num_sampling_steps = GENERATE_NUM_SAMPLING_STEPS
    if GENERATE_PAIRWISE is not None:
        args.pairwise = GENERATE_PAIRWISE

    if AUTO_TUNE and torch.cuda.is_available():
        try:
            from util.auto_tune import auto_tune
            args.batch_size = auto_tune(method="table", model_name=MODEL, img_size=IMG_SIZE,
                                        reserve=TUNE_RESERVE, max_batch=TUNE_MAX_BATCH,
                                        max_gen_bsz=TUNE_MAX_GEN_BSZ,
                                        batch_size_fallback=args.batch_size,
                                        verbose=False)["gen_bsz"]
        except Exception:
            pass

    print("[Cell4] 推理生成中 ... batch_size:", args.batch_size)
    gen_main(args)
else:
    print("[Cell4] DO_GENERATE=False，跳过")

## Cell 5 · 缺失字补集生成（类似 HanziGen 的缺字补全）

**本 Cell 做什么（无需修改）：**

1. 用 fontTools 读取目标字体的 cmap，计算 **CHARSET 字符集 − 目标字体已覆盖字符 = 缺失字**
2. 缺失字须能被参照字体渲染（模型以参照字形为 content 输入）
3. 样式参考图来自**目标字体自身**（与训练时的 ref 网格一致）
4. 用训练好的 checkpoint 逐个生成缺失字 PNG，并输出 `missing_chars.txt` 清单

**产物：** `outputs/<字体>/missing_chars/`

- `generated/U+XXXX.png`：补全的缺失字
- `compare/U+XXXX.png`：`源字形 | 生成结果` 对比图
- `missing_chars.txt`：缺失字清单（U+XXXX 与字符对照）

> 用法示例：字体缺某些简/繁体字时一键补齐。**提示**：想让字符标签与预训练对齐、补集效果最好，
> 训练时建议 `TRAIN_CHARS_PER_FONT` 覆盖整个 CHARSET、`MAX_CHARS_PER_FONT=None`。

In [ ]:
# ============================================================
# Cell 5 · 缺失字补集（一般不用改）
# ============================================================
if DO_MISSING_GEN:
    ckpt = os.path.join(OUTPUT_DIR, "checkpoint-last.pth")
    assert os.path.exists(ckpt), "[Cell5] 未找到 checkpoint: %s" % ckpt

    target = TARGET_FONTS[0]
    target_path = target if os.path.isabs(target) else os.path.join(PROJECT_ROOT, target)

    cmd = [sys.executable, os.path.join(PROJECT_ROOT, "scripts", "generate_missing_chars.py"),
           "--checkpoint", ckpt,
           "--target-font", target_path,
           "--source-font", SOURCE_FONT_PATH,
           "--charset", MISSING_CHARSET or CHARSET,
           "--output-dir", MISSING_OUTPUT_DIR,
           "--batch-size", str(MISSING_BATCH_SIZE or 32),
           "--pairwise", MISSING_PAIRWISE or "none",
           "--seed", str(SEED),
           "--device", "auto"]
    if MISSING_NUM_IMAGES is not None:
        cmd += ["--num-images", str(MISSING_NUM_IMAGES)]
    if MISSING_CFG is not None:
        cmd += ["--cfg", str(MISSING_CFG)]
    if MISSING_SAMPLING_METHOD is not None:
        cmd += ["--sampling-method", MISSING_SAMPLING_METHOD]
    if MISSING_NUM_SAMPLING_STEPS is not None:
        cmd += ["--num-sampling-steps", str(MISSING_NUM_SAMPLING_STEPS)]
    if MISSING_REF_CHARS:
        cmd += ["--ref-chars", MISSING_REF_CHARS]

    print("[Cell5] 命令:", " ".join(cmd))
    subprocess.run(cmd, check=True)
else:
    print("[Cell5] DO_MISSING_GEN=False，跳过")

## Cell 6 · 导出打包 + 浏览器下载

**本 Cell 做什么（无需修改）：**

- 把 Cell 4 的 `generated_chars/` 与 Cell 5 的 `missing_chars/` 下所有 PNG
  （可选含 LoRA checkpoint）打包成 zip
- 保存到 `exports/`（可视化目录，可在左侧文件树下载），并给出**浏览器一键下载**按钮
- zip 超过 1 GB 时不再生成 base64 下载链接，请直接从文件树下载

In [ ]:
# ============================================================
# Cell 6 · 导出（一般不用改）
# ============================================================
if DO_EXPORT:
    roots = [GEN_OUTPUT_DIR]
    if os.path.isdir(MISSING_OUTPUT_DIR):
        roots.append(MISSING_OUTPUT_DIR)
    if not any(os.path.isdir(r) for r in roots):
        print("[Cell6] 未找到生成目录，跳过:", roots)
    else:
        os.makedirs(EXPORTS_DIR, exist_ok=True)
        stamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
        zip_name = "%s_%s_%s.zip" % (EXPORT_PREFIX, FONT_TAG, stamp)
        zip_path = os.path.join(EXPORTS_DIR, zip_name)

        count = 0
        with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
            for root_dir in roots:
                for root, _, files in os.walk(root_dir):
                    for f in sorted(files):
                        if f.lower().endswith((".png", ".jpg", ".jpeg")):
                            full = os.path.join(root, f)
                            zf.write(full, os.path.relpath(full, PROJECT_ROOT))
                            count += 1
            if EXPORT_INCLUDE_CHECKPOINT:
                ck = os.path.join(OUTPUT_DIR, "checkpoint-last.pth")
                if os.path.exists(ck):
                    zf.write(ck, "checkpoint/checkpoint-last.pth")

        size_mb = os.path.getsize(zip_path) / 1024 / 1024
        print("[Cell6] 导出完成: %s（%d 张图片, %.1f MB）" % (zip_path, count, size_mb))

        if size_mb < 1000:
            with open(zip_path, "rb") as f:
                b64 = base64.b64encode(f.read()).decode()
            from IPython.display import HTML, display
            display(HTML(
                '<a href="data:application/zip;base64,%s" download="%s" '
                'style="font-size:18px;background:#0d6efd;color:#fff;'
                'padding:10px 20px;text-decoration:none;border-radius:6px">'
                '&#128229; 点击下载 %s（%.1f MB）</a>'
                % (b64, zip_name, zip_name, size_mb)))
        else:
            print("[Cell6] zip 较大，请直接在左侧文件树 exports/ 目录下载: %s" % zip_path)
else:
    print("[Cell6] DO_EXPORT=False，跳过")

## 完成！

**结果一览：**

| 产物 | 位置 |
|---|---|
| 常规推理 PNG | `outputs/<字体>/generated_chars/` |
| 缺失字补集 PNG + `missing_chars.txt` | `outputs/<字体>/missing_chars/` |
| LoRA checkpoint | `outputs/<字体>/checkpoint-last.pth` |
| 导出 zip | `exports/` |

**调参速查：**

- 生成效果不理想：先调 `GENERATE_CFG`（加大）→ `GENERATE_SAMPLING_METHOD="heun"` → 加步数
- 风格化明显（行书/草书/手写）：`LORA_R=64`、`EPOCHS=300+`、`CFG=3.5~4.0`
- 补集效果不佳：训练时用完整 CHARSET（`TRAIN_CHARS_PER_FONT` ≥ 字符集大小、`MAX_CHARS_PER_FONT=None`）
- 显存相关：保持 `AUTO_TUNE=True`，自动算 batch_size，无需手动猜